<a href="https://colab.research.google.com/github/SabirDivs/colab-utils/blob/main/pdf_watermark_remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pikepdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.7 MB/s eta 0:00:00


In [2]:
# ==============================================================
# PDF Unlocker + Text Watermark Remover (Google Colab)
# ==============================================================
# 1. Unlocks password‑protected or restricted PDFs.
# 2. Optionally removes a text watermark if you know the exact text.
#
# Works well for watermarks added as real text (e.g., “DRAFT”).
# For image‑based watermarks, a different approach would be needed.
# ==============================================================

import pikepdf
from pikepdf import Pdf, Page, ContentStreamInstruction
from google.colab import files
import io, re

print("📂 Upload your password‑protected/restricted PDF file:")
uploaded = files.upload()

if not uploaded:
    print("No file uploaded. Exiting.")
else:
    original_name = next(iter(uploaded))
    pdf_bytes = uploaded[original_name]

    # --- 1. Unlock the PDF (remove open password & owner restrictions) ---
    password = ''
    while True:
        try:
            pdf = Pdf.open(io.BytesIO(pdf_bytes), password=password)
            break
        except pikepdf.PasswordError:
            password = input("🔑 Enter password (or press Enter if none): ")
        except Exception as e:
            print(f"❌ Cannot open PDF: {e}")
            raise SystemExit

    # --- 2. Check if user wants to remove a text watermark ---
    remove_wm = input("Do you want to try removing a TEXT watermark? (y/n): ").strip().lower()
    if remove_wm == 'y':
        # Show text from the first page so the user can identify the watermark
        first_page = pdf.pages[0]
        # Extract text using pikepdf's text extraction (simple, may not be perfect)
        text_page0 = ""
        try:
            # pikepdf >=5 has .extract_text(), but for compatibility we use a helper
            from pikepdf import Page
            # The method is available on Page objects since pikepdf 5.0
            text_page0 = first_page.extract_text() or ""
        except Exception:
            # fallback: use a minimal extraction via content stream if needed
            pass

        print("\n📄 Text found on page 1 (may include watermark):")
        print("-------------------------------------------------")
        print(text_page0[:2000])  # first 2000 chars – adjust if needed
        print("-------------------------------------------------")

        watermark_text = input("Enter the EXACT watermark text to remove (case‑sensitive): ").strip()
        if watermark_text:
            print(f"🔍 Searching and removing all instances of “{watermark_text}” from every page...")

            # --- Remove the text watermark from each page’s content stream ---
            for page_num, page in enumerate(pdf.pages, start=1):
                # Parse the page’s content into a list of instructions
                instructions = pikepdf.parse_content_stream(page)
                new_instructions = []
                i = 0
                while i < len(instructions):
                    instr = instructions[i]
                    # A text‑showing operator (Tj, TJ, ', ") will be preceded by font selection
                    # We look for sequences where the operand is our watermark string.
                    # In PDF content streams, text strings are often written as (DRAFT) Tj
                    # or as an array in TJ. We'll handle both.
                    if instr.operator in (pikepdf.Operator('Tj'), pikepdf.Operator('TJ')):
                        operands = instr.operands
                        if operands:
                            text_param = operands[0]
                            text_to_check = ""
                            if isinstance(text_param, pikepdf.String):
                                text_to_check = str(text_param)  # pikepdf.String returns bytes -> decode
                            elif isinstance(text_param, pikepdf.Array):
                                # For TJ arrays, join the strings
                                parts = []
                                for item in text_param:
                                    if isinstance(item, pikepdf.String):
                                        parts.append(str(item))
                                    # skip numeric kerning adjustments
                                text_to_check = "".join(parts)
                            # If this instruction contains the watermark, skip it
                            if watermark_text in text_to_check:
                                # Skip this instruction – do not add to new_instructions
                                pass
                            else:
                                new_instructions.append(instr)
                        else:
                            new_instructions.append(instr)
                    else:
                        new_instructions.append(instr)
                    i += 1
                # Replace the page’s content with the cleaned instructions
                page.contents_coalesce()  # ensure single content stream
                new_stream = pikepdf.unparse_content_stream(new_instructions)
                page.Contents = pdf.make_stream(new_stream)

            print("✅ Watermark text removed from all pages (where found).")
        else:
            print("No watermark text entered. Skipping removal.")
    else:
        print("Skipping watermark removal.")

    # --- 3. Save and download the unlocked (and cleaned) PDF ---
    output_name = f"unlocked_nowatermark_{original_name}"
    out_buffer = io.BytesIO()
    pdf.save(out_buffer)
    out_buffer.seek(0)
    files.download(out_buffer, output_name)
    print(f"✅ Done! Your file is saved as “{output_name}”. It has no restrictions and text watermarks have been removed if requested.")

📂 Upload your password‑protected/restricted PDF file:


Saving 2867-12th-Class-English-Unit-1-n-2-(PECTAA).pdf to 2867-12th-Class-English-Unit-1-n-2-(PECTAA).pdf
Do you want to try removing a TEXT watermark? (y/n): y

📄 Text found on page 1 (may include watermark):
-------------------------------------------------

-------------------------------------------------
Enter the EXACT watermark text to remove (case‑sensitive): 
No watermark text entered. Skipping removal.


TypeError: download() takes 1 positional argument but 2 were given